<a href="https://colab.research.google.com/github/lucaprohor/Banking-Application-/blob/main/Secure_Banking_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install bcrypt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.5 MB/s eta 0:00:00


In [ ]:
import json
import os
import getpass
import time
import re
import random
import bcrypt


In [ ]:
DATABASE_FILE = 'banking_data.json'
TRANSACTION_LOG_FILE = 'transaction_log.txt'

if os.path.exists(DATABASE_FILE):
    with open(DATABASE_FILE, 'r') as file:
        users = json.load(file)
else:
    users = {}

if not os.path.exists(TRANSACTION_LOG_FILE):
    open(TRANSACTION_LOG_FILE, 'w').close()

def save_data():
    with open(DATABASE_FILE, 'w') as file:
        json.dump(users, file)

In [ ]:
def validate_username(username):
    return re.match(r"^[a-zA-Z0-9]{5,15}$", username) is not None


def validate_password(password):
    return (
        len(password) >= 8
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"[0-9]", password)
        and re.search(r"[\W_]", password)
    )

In [ ]:
def hash_password(password):
    encrypted = bcrypt.hashpw(password.encode(),bcrypt.gensalt())
    return encrypted

In [ ]:
def create_account(username, password):
    if not validate_username(username):
        print("Invalid username. Use 5-15 letters and numbers.")
        return None

    if username in users:
        print("Username already exists.")
        return None

    if not validate_password(password):
        print("Password must contain at least 8 characters, including uppercase, lowercase, number, and special character.")
        return None

    existing_numbers ={
        account['account_number']
        for account in users.values()
        if 'account_number' in account
    }

    account_num = random.randint(1, 999999999999999999)

    while account_num in existing_numbers:
        account_num = random.randint(1, 999999999999999999)

    account ={
        'username': username,
        'password': hash_password(password).decode('utf-8'),
        'account_number': account_num,
        'balance': 0.0,
        'login_attempts': 0,
        'is_blocked': False,
        'blocked_until': None
    }

    users[username] = account
    save_data()

    print(f"Account created successfully for {username}.")
    return account

In [ ]:
def authenticate_user(username):
    if username not in users:
        print("Username not found.")
        return "account does not exist with this username"
    user_data = users[username]
    if user_data['is_blocked']:
        if time.time() < user_data['blocked_until']:
            wait_time = int(user_data['blocked_until'] - time.time())
            print(f"Your account is temporarily blocked. Please wait {wait_time} seconds before trying again.")
            return None
    stored_password = user_data['password'].encode()

    if bcrypt.checkpw(password.encode(), stored_password):
        user_data['login_attempts'] = 0
        save_data()
        return "Login successful."

    else:
        user_data['login_attempts'] += 1

        lockout_message = account_lockout(username)

        if lockout_message:
            return lockout_message

        remaining = 3 - user_data['login_attempts']
        save_data()

        return f"Invalid password. {3 - user_data['login_attempts']} attempts remaining."

In [ ]:
def save_data():
    with open(DATABASE_FILE, 'save room') as file:
        json.dump(users, file)

In [ ]:
def account_lockout(username):
    user_data = users[username]

    if user_data['login_attempts'] >= 3:
        user_data['is_blocked'] = True
        user_data['blocked_until'] = time.time() + 600
        save_data()

        return "Your account has been temporarily blocked for 10 minutes."

    return None

In [ ]:

def login(username, password):
    if username not in users:
        return "Username not found."

    user_data = users[username]


    if user_data['is_blocked']:
        if time.time() < user_data['blocked_until']:
            wait_time = int(user_data['blocked_until'] - time.time())
            return f"Your account is temporarily blocked. Please wait {wait_time} seconds before trying again."
        else:
            user_data['is_blocked'] = False
            user_data['login_attempts'] = 0
            user_data['blocked_until'] = None
            save_data()

    stored_password = user_data['password'].encode()

    if bcrypt.checkpw(password.encode(), stored_password):
        user_data['login_attempts'] = 0
        save_data()
        return "Login successful."

    else:
        user_data['login_attempts'] += 1

        lockout_message = account_lockout(username)

        if lockout_message:
            return lockout_message

        remaining = 3 - user_data['login_attempts']
        save_data()

        return f"Invalid password. {3 - user_data['login_attempts']} attempts remaining."

In [ ]:
def log_transaction(username, transaction):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

    log_entry = f"{timestamp} - {username}: {transaction}\n"

    with open(TRANSACTION_LOG_FILE, 'a') as file:
        file.write(log_entry)

def check_balance(username):
  print(f"Current balance: ${users[username]['balance']:.2f}")


def deposit(username):
    try:
        amount = float(input("Enter deposit amount: $"))

        if amount <= 0:
            print("Amount must be positive.")
            return None

        users[username]['balance'] += amount

        log_transaction(username, f"Deposit: +${amount:.2f}")
        save_data()

        print(f"${amount:.2f} deposited successfully.")

    except ValueError:
        print("Please enter a valid numeric amount.")
def withdraw(username, amount):
    try:
        if amount <= 0:
            print("Amount must be positive.")
            return None

        if users[username]['balance'] < amount:
            print("Insufficient funds.")
            return None

        users[username]['balance'] -= amount

        log_transaction(username, f"Withdraw: -${amount:.2f}")
        save_data()

        print(f"${amount:.2f} withdrawn successfully.")

    except ValueError:
        print("Please enter a valid numeric amount.")

def transfer(sender, reciver, amount):
    if sender not in users or reciver not in users:
        print("Invalid sender or reciver.")
        return None
    if sender == reciver :
        print("sender and reciver can not be the same user")
        return None
    try:
        if amount <= 0:
            print("Amount must be positive.")
            return None

        if users[sender]['balance'] < amount:
            print("Insufficient funds.")
            return None

        users[sender]['balance'] -= amount
        users[reciver]['balance'] += amount

        log_transaction(
            sender,
            f"Transfer to {reciver}: -${amount:.2f}"
        )
        log_transaction(
            reciver,
            f"Transfer from {sender}: +${amount:.2f}"
        )

        save_data()
        print(f"${amount:.2f} transferred to {reciver}.")

    except ValueError:
        print("Please enter a valid numeric amount.")

In [ ]:
def user_navigation(username):
    while True:
        print("\nBanking Menu")
        print("1. Deposit")
        print("2. Withdraw")
        print("3. Check Balance")
        print("4. Transfer")
        print("5. Logout")

        choice = input("Choose an option: ")

        if choice == '1':
            deposit(username)

        elif choice == '2':
            try:
                amount = float(input("Enter withdrawal amount: $"))
                withdraw(username, amount)
            except ValueError:
                print("Please enter a valid numeric amount.")

        elif choice == '3':
            check_balance(username)

        elif choice == '4':
            receiver = input("Enter recipient username: ")
            try:
                amount = float(input("Enter transfer amount: $"))
                transfer(username, receiver, amount)
            except ValueError:
                print("Please enter a valid numeric amount.")

        elif choice == '5':
            print("Logging out...")
            break

        else:
            print("Invalid choice. Please try again.")


def main():
    while True:
        print("\nMain Menu")
        print("1. Create Account")
        print("2. Login")
        print("3. Exit")

        choice = input("Choose an option: ")

        if choice == '1':
            username = input("Enter a username: ")
            password = getpass.getpass("Enter a password: ")
            confirm_password = getpass.getpass("Confirm password: ")

            if password != confirm_password:
                print("Passwords do not match.")
                continue

            create_account(username, password)

        elif choice == '2':
            username = input("Enter your username: ")
            password = getpass.getpass("Enter your password: ")

            result = login(username, password)
            print(result)

            if result == "Login successful.":
                banking_menu(username)

        elif choice == '3':
            print("Exiting the system. Goodbye!")
            break

        else:
            print("Invalid choice. Please try again.")


main()


Main Menu
1. Create Account
2. Login
3. Exit
Choose an option: 3
Exiting the system. Goodbye!


In [ ]:
test_username = "TestUser2"
test_password = "BankTest124!"

account = create_account(test_username, test_password)

if account:
    print("Account number:", account['account_number'])
    print("Login test:", login(test_username, test_password))

In [ ]:
test_username = "TestUser3"
test_password = "BankTest125!"
test_username2 = "TestUser2"
test_password2 = "BankTest124!"

account = create_account(test_username2, test_password2)
account = create_account(test_username, test_password)
check_balance(test_username)
withdraw(test_username, 100)
transfer(test_username, test_username2, 1000)
user_navigation(test_username)
save_data()



Username already exists.
Username already exists.
Current balance: $460689.00
$100.00 withdrawn successfully.
$1000.00 transferred to TestUser2.

Banking Menu
1. Deposit
2. Withdraw
3. Check Balance
4. Transfer
5. Logout
Choose an option: 2
Enter withdrawal amount: $678
$678.00 withdrawn successfully.

Banking Menu
1. Deposit
2. Withdraw
3. Check Balance
4. Transfer
5. Logout
Choose an option: 2
Enter withdrawal amount: $450
$450.00 withdrawn successfully.

Banking Menu
1. Deposit
2. Withdraw
3. Check Balance
4. Transfer
5. Logout
Choose an option: 1
Enter deposit amount: $10000
$10000.00 deposited successfully.

Banking Menu
1. Deposit
2. Withdraw
3. Check Balance
4. Transfer
5. Logout
Choose an option: 3
Current balance: $468461.00

Banking Menu
1. Deposit
2. Withdraw
3. Check Balance
4. Transfer
5. Logout
Choose an option: 4
Enter recipient username: TestUser2
Enter transfer amount: $35
$35.00 transferred to TestUser2.

Banking Menu
1. Deposit
2. Withdraw
3. Check Balance
4. Transfe